# Multi-Object Tracking with LibreYOLO

LibreYOLO ships ByteTrack-based multi-object tracking on top of any of its detection families (YOLOv9, YOLOX, RF-DETR). This notebook walks through:
1. The raw `ByteTracker` API on synthetic detections (no network, no weights).
2. End-to-end `model.track(video_path)` on a real video.
3. Tracker config knobs (`track_high_thresh`, `track_buffer`, etc.) and what they do.

The first half runs on CPU in <5 seconds. The second half needs a detection checkpoint.

## 1. Install

In [ ]:
# pip install -e git+https://github.com/aalvsz/libreyolo@agentic/tracking-pipeline#egg=libreyolo
import numpy as np
import torch
from libreyolo.tracking import ByteTracker, TrackConfig
from libreyolo.utils.results import Boxes, Results

## 2. Raw tracker on synthetic detections

`ByteTracker.update(Results)` takes a `Results` of detections for one frame and returns a new `Results` with `track_id` set. Any detector that produces `Results` can drive it.

In [ ]:
def make_frame(boxes_xyxy, scores, cls):
    if len(boxes_xyxy) == 0:
        xyxy = torch.zeros((0, 4)); conf = torch.zeros((0,)); c = torch.zeros((0,))
    else:
        xyxy = torch.tensor(boxes_xyxy, dtype=torch.float32)
        conf = torch.tensor(scores, dtype=torch.float32)
        c = torch.tensor(cls, dtype=torch.float32)
    return Results(boxes=Boxes(boxes=xyxy, conf=conf, cls=c),
                   orig_shape=(480, 640), names={0: 'obj'})

tracker = ByteTracker(minimum_consecutive_frames=1, track_high_thresh=0.3, new_track_thresh=0.3)

# An object sliding right across 5 frames
for dx in range(0, 100, 20):
    r = tracker.update(make_frame([[100+dx, 150, 200+dx, 250]], [0.9], [0]))
    print(f'dx={dx:>3}   track_ids={r.track_id.tolist()}   boxes={r.boxes.xyxy.tolist()[0]}')

The same ID persists across all five frames. That's ByteTrack's first-stage association at work — a single high-confidence detection matched to the predicted Kalman state of the existing track.

## 3. Surviving short occlusions

If the detector drops the object for a few frames, ByteTrack extrapolates via Kalman filter and re-associates when it reappears — keeping the same track ID.

In [ ]:
tracker = ByteTracker(minimum_consecutive_frames=1, track_high_thresh=0.3,
                      new_track_thresh=0.3, track_buffer=30)
# Frame 0: track starts
r0 = tracker.update(make_frame([[100, 100, 200, 200]], [0.9], [0]))
print('frame 0:', r0.track_id.tolist())
# Frames 1-2: object missing (empty detection)
for _ in range(2): tracker.update(make_frame([], [], []))
# Frame 3: reappears
r3 = tracker.update(make_frame([[140, 110, 240, 210]], [0.9], [0]))
print('frame 3:', r3.track_id.tolist(), '(same ID — track recovered)')

## 4. End-to-end on a real video

`model.track(source)` is a generator yielding one `Results` per frame. Use `save=True` to write an annotated output video.

In [ ]:
from libreyolo import LibreYOLO

# Assumes you have a video. See scripts/track_yolo_video.py for a full CLI.
# model = LibreYOLO('LibreYOLO9t.pt')   # auto-downloads COCO-pretrained weights
# for result in model.track('path/to/input.mp4',
#                           track_conf=0.25, iou=0.45,
#                           save=True, output_path='runs/track/out.mp4'):
#     # result has boxes + track_id for every confirmed track in this frame
#     if result.track_id is not None and len(result.boxes) > 0:
#         print(f'tracks this frame: {result.track_id.tolist()}')

## 5. What knobs matter

| Param | Default | When to change |
|---|---|---|
| `track_high_thresh` | 0.25 | Raise for cleaner tracks (fewer false starts); lower if objects are often low-conf |
| `track_low_thresh` | 0.1 | Lower bound for ByteTrack's second-stage recovery. Keep below `high_thresh`. |
| `new_track_thresh` | 0.25 | Minimum conf to START a new track (vs keeping an existing one alive) |
| `match_thresh` | 0.8 | IoU cost threshold for first-stage matching. Tighten on small/crowded objects. |
| `track_buffer` | 30 | Frames a lost track stays eligible for recovery. Increase for long occlusions. |
| `frame_rate` | 30 | Used to scale `track_buffer` in absolute time. Match your video. |
| `minimum_consecutive_frames` | 1 | Frames required before a track is "confirmed" (emits a track_id). Raise to suppress flickers. |

See `scripts/track_yolo_video.py` for a full CLI that exposes all of these.